# elbo-loss-sum-with-beta — faded example 2: Fill the diagonal-Gaussian KL term

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `elbo-loss-sum-with-beta`. Running the beacon reports progress on the `VAE: ELBO loss sum with beta` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `VAE: ELBO loss sum with beta` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`elbo-loss-sum-with-beta`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "elbo-loss-sum-with-beta"
DD_SUBTOPIC = "VAE: ELBO loss sum with beta"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The KL of a diagonal Gaussian posterior `N(mu, sigma^2)` from the standard normal prior has the closed form `-0.5 * sum(1 + 2*logsigma - mu^2 - exp(2*logsigma))` over the latent dimension. This scalar is then weighted by `beta` and added to the reconstruction loss.

## Faded exercise 2

Implement `vae_loss(x, x_hat, mu, logsigma, beta)`. Compute the batch-mean MSE reconstruction, the batch-mean closed-form diagonal-Gaussian KL, and return `recon + beta * kl`. Complete the blanked KL computation (per-sample sum over the latent axis, then batch mean).

**Fill in:** the closed-form diagonal-Gaussian KL, summed over the latent axis then averaged over the batch

In [ ]:
import torch as t

t.manual_seed(4)
x = t.randn(8, 16)
x_hat = x + 0.1 * t.randn(8, 16)
mu = t.randn(8, 4)
logsigma = t.randn(8, 4)

def vae_loss(x, x_hat, mu, logsigma, beta):
    recon = ((x_hat - x) ** 2).mean()
    kl = None  # TODO: the closed-form diagonal-Gaussian KL, summed over the latent axis then averaged over the batch
    return recon + beta * kl

print(float(vae_loss(x, x_hat, mu, logsigma, 1.0)))


def _test():
    out = vae_loss(x, x_hat, mu, logsigma, 1.0)
    # independent ground truth: per-element KL formula, different reduction order
    sigma2 = (2 * logsigma).exp()
    kl_elem = -0.5 * (1 + 2 * logsigma - mu ** 2 - sigma2)
    kl_ref = kl_elem.sum(dim=1).mean()
    recon_ref = t.mean((x_hat - x) ** 2)
    expected = recon_ref + 1.0 * kl_ref
    assert t.allclose(out, expected, atol=1e-5), (float(out), float(expected))
    # KL must be non-negative for these standard-normal-prior stats only if posterior near prior;
    # instead assert the composite exceeds the bare reconstruction when kl>0
    assert float(kl_ref) > 0
    assert float(out) > float(recon_ref)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(4)
x = t.randn(8, 16)
x_hat = x + 0.1 * t.randn(8, 16)
mu = t.randn(8, 4)
logsigma = t.randn(8, 4)

def vae_loss(x, x_hat, mu, logsigma, beta):
    recon = ((x_hat - x) ** 2).mean()
    kl = (-0.5 * (1 + 2 * logsigma - mu ** 2 - (2 * logsigma).exp()).sum(dim=1)).mean()
    return recon + beta * kl

print(float(vae_loss(x, x_hat, mu, logsigma, 1.0)))
```
</details>